In [4]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')


import torch
import os
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer
from dictionary_learning.mask_scae import SCAESuite, MergedSCAESuite # Your SCAESuite class
from datasets import load_dataset
from dictionary_learning.buffer import chunk_and_tokenize
from interp.interp_utils_new import (
    load_tokenized_dataset,
    collect_activations_and_tokens,
    generate_feature_dashboard
)

torch.set_grad_enabled(False)

# --- Configuration ---
MODEL_NAME = "EleutherAI/pythia-70m"
PATH_TO_PILE = "/root/dictionary_learning/pile-uncopyrighted"
# For local pile, you might need to specify data_files if it's a collection of .jsonl.gz
# e.g., data_files = {"train": ["/path/to/pile/train/00.jsonl.gz", ...]}
# For HF streaming, data_files can be None.
DATA_FILES = None # Set to list of local file paths if using local pile and not streaming
STREAM_PILE = True # If True, streams from HF. If False and DATA_FILES is None, loads non-streamed from HF.
                   # If DATA_FILES is set, STREAM_PILE=True will stream from those local files.
SEQ_LEN = 128
NUM_SAMPLES_TO_LOAD = 1000 # How many sequences to prepare from the dataset
NUM_SAMPLES_TO_PROCESS = 1000 # How many sequences to run through the suite for activations (max_batches_to_process * batch_size)
BATCH_SIZE_COLLECT = 64
OUTPUT_ACTIVATIONS_DIR = "output_activations_pile"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# --- Initialize Model and Tokenizer ---
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Load or Create your SCAESuite ---
# Option 1: Load from a checkpoint (replace with your actual loading logic)
# suite = SCAESuite.from_pretrained(SUITE_REPO_ID, model, device=DEVICE, dtype=torch.float32)

# Option 2: Initialize a new suite (for testing the pipeline if you don't have a trained one)
# This is a placeholder, ensure parameters match your actual suite setup
suite = SCAESuite.from_pretrained(
    repo_id="jacobcd52/pythia-70m_mask0_fact_fvu0_fvu_sparse0_fvu1.0_lr0.0005",
    model=model,
    device=DEVICE,
)

print(f"Using device: {DEVICE}")
os.makedirs(OUTPUT_ACTIVATIONS_DIR, exist_ok=True)

Loaded pretrained model EleutherAI/pythia-70m into HookedTransformer
Using device: cuda


In [3]:
raw_dataset = load_dataset(
    PATH_TO_PILE,
    split=f"train[:10%]",
)

raw_dataset = raw_dataset.shuffle(seed=42)


tokenized_dataset = chunk_and_tokenize(
    dataset=raw_dataset,
    tokenizer=tokenizer,
    text_key="text",
    max_length=SEQ_LEN,
    num_proc=max(1, os.cpu_count() // 2),
    load_from_cache_file=True
)

if len(tokenized_dataset) > NUM_SAMPLES_TO_PROCESS:
    dataset_to_process = tokenized_dataset.select(range(NUM_SAMPLES_TO_PROCESS))
else:
    dataset_to_process = tokenized_dataset


# Quick check of a sample
sample = next(iter(dataset_to_process))
if isinstance(sample["input_ids"], torch.Tensor):
    print("Sample token IDs shape:", sample["input_ids"].shape)
else: # It's a list for streaming datasets before DataLoader
    print("Sample token IDs length (list):", len(sample["input_ids"]))
    # If you need it as a tensor for this check:
    # sample_tensor = torch.tensor(sample["input_ids"])
    # print("Sample token IDs shape (converted to tensor):", sample_tensor.shape)

# Decoding should still work fine as tokenizer.decode can handle lists of IDs
print("Sample tokens (decoded):", tokenizer.decode(sample["input_ids"]))

Generating train split:   0%|          | 0/5899215 [00:00<?, ? examples/s]

Filter (num_proc=64): 100%|██████████| 589922/589922 [00:01<00:00, 505967.72 examples/s]


Sample token IDs shape: torch.Size([128])
Sample tokens (decoded): "The driver-based software compiles user data, provides a range of statistics for analysis, and hands out trophies when exceptional milestones have been reached. It’s an entertaining way for gamers to keep track of their mouse skills – and it even lets players share their accomplishments with others via social network sites, like Facebook."

And now was acknowledged the presence of the Red Death. He had come like a thief in the night. And one by one dropped the revellers in the blood-bedewed halls of their revel, and died each in the despairing posture of his fall. And the life of the ebony


In [46]:
toks_list = []
iterable_dataset = iter(dataset_to_process)
for _ in range(128):
    toks_list.append(next(iterable_dataset)['input_ids'])
toks = torch.stack(toks_list[:64], dim=0)
toks_test = torch.stack(toks_list[64:], dim=0)

layer = 3
ae = suite.module_dict[f'attn_{layer}'].ae
ae.k = 64
hook_pt = f'blocks.{layer}.hook_attn_out'
_, cache = model.run_with_cache(toks, return_type="loss", names_filter=[hook_pt])
act = cache[hook_pt]
f = ae.encode(act)
recons = ae.decode(f)
fvu = (act - recons).pow(2).sum() / (act - act.mean(dim=[0, 1])).pow(2).sum()

print("SAE FVU: ", fvu)

_, cache_test = model.run_with_cache(toks_test, return_type="loss", names_filter=[hook_pt])
act_test = cache_test[hook_pt]

# Compute PCA on training activations
q = 100
U, S, V = torch.pca_lowrank(act.reshape(-1, act.shape[-1]), q=q)
# Project test activations onto PCA components
act_test_flat = act_test.reshape(-1, act_test.shape[-1])
act_test_proj = act_test_flat @ V
# Compute variance explained
total_var = (act_test_flat - act_test_flat.mean(dim=[0, 1])).pow(2).sum()
explained_var = (act_test_proj - act_test_proj.mean(dim=[0, 1])).pow(2).sum()
print(f"FVU of top {q} PCA components: {1 - explained_var/total_var:.3f}")

SAE FVU:  tensor(0.0813, device='cuda:0')
FVU of top 100 PCA components: 0.207


In [7]:
# --- Collect Activations ---
# Determine the number of batches to process
max_batches = NUM_SAMPLES_TO_PROCESS // BATCH_SIZE_COLLECT

# Collect for sparse_false mode
print("Starting activation collection for sparse_false mode...")
collect_activations_and_tokens(
    suite=suite,
    model=model,
    tokenizer=tokenizer,
    dataset=dataset_to_process,
    device=DEVICE,
    output_dir=OUTPUT_ACTIVATIONS_DIR,
    run_mode_sparse=False,
    batch_size=BATCH_SIZE_COLLECT,
    max_batches_to_process=max_batches
)
print("Finished activation collection for sparse_false mode.")

# # Collect for sparse_true mode
# print("\\nStarting activation collection for sparse_true mode...")
# collect_activations_and_tokens(
#     suite=suite,
#     model=model,
#     tokenizer=tokenizer,
#     dataset=dataset_to_process,
#     device=DEVICE,
#     output_dir=OUTPUT_ACTIVATIONS_DIR,
#     run_mode_sparse=True,
#     batch_size=BATCH_SIZE_COLLECT,
#     max_batches_to_process=max_batches
# )
# print("Finished activation collection for sparse_true mode.")


Starting activation collection for sparse_false mode...
Starting activation collection. Mode: Non-sparse
Saving to: output_activations_pile/sparse_false
Processed and saved batch 0
Processed and saved batch 10
Reached max_batches_to_process: 15. Stopping.
Finished activation collection for mode: Non-sparse.
Finished activation collection for sparse_false mode.


In [48]:
# --- Generate Feature Dashboard ---
# Choose which activations to analyze (sparse or non-sparse)
mode_to_analyze = "sparse_false" # or "sparse_false"
activations_path = os.path.join(OUTPUT_ACTIVATIONS_DIR, mode_to_analyze)

# Specify the module and feature index you want to inspect
# Example: first attention module (attn_0), feature index 123
# You'll need to know the valid module names and feature ranges for your suite
module_to_inspect = "attn_3"


# Check if the activations path for the chosen mode exists
if not os.path.exists(activations_path):
    print(f"Activations path {activations_path} does not exist. Run collection for this mode first.")
else:
    # Check if there are any batch folders in the activations_path
    batch_folders_exist = any(d.startswith("batch_") for d in os.listdir(activations_path))
    if not batch_folders_exist:
        print(f"No batch data found in {activations_path}. Ensure activation collection was successful.")
    else:
        print(f"Generating dashboard using activations from: {activations_path}")
        generate_feature_dashboard(
            module_name_str=module_to_inspect,
            feature_idx_in_module=0,
            activations_base_dir=activations_path,
            tokenizer=tokenizer,
            model=model,
            suite=suite,
            k_top_contexts=20,
            context_window_size=256 # Number of tokens around the max activating one
        )

Generating dashboard using activations from: output_activations_pile/sparse_false
Generating dashboard for: attn_3, Feature Index: 0


In [50]:
s_list = [
"""The educational aim of the proposed K01 proposal is to allow the applicant to train in developmental affective neuroscience and pediatric bipolar disorder and acquire the skills necessary to characterize neurodevelopmental abnormalities in neural systems of emotion regulation in young adolescents at high genetic risk of bipolar disorder. Bipolar disorder is a chronic and debilitating psychiatric disorder in adults and even more so in children and adolescents. It is characterized by significant impairments in emotion regulation, which have been associated with functional abnormalities in prefrontal and subcortical neural regions. Bipolar disorder may be mediated by neurodevelopmental abnormalities in these neural regions. The onset of bipolar disorder increases dramatically in adolescence,""",
"""Zena Sutherland<br><br>Zena Sutherland (1915 - June 12, 2002) was an American reviewer of children's literature.  She is best known for her contributions to the Bulletin of the Center for Children's Books and as the author of the library science textbook, Children and Books.<br><br>Early life and education<br>Sutherland was born in Winthrop, Massachusetts in 1915 but was raised in Chicago by her mother after her parents’ divorce. She graduated from the University of Chicago in 1937. In 1966, she received her master's, also from the University of Chicago, in library science.<br><br>""",
"""1754 in science<br><br>The year 1754 in science and technology involved some significant events.<br><br>Astronomy<br> Immanuel Kant, German philosopher, postulates retardation of Earth's orbit.<br><br>Chemistry<br> Joseph Black, Scottish chemist, discovers carbonic acid gas.<br><br>Earth sciences<br> Albert Brahms, Frisian Dijkgraaf, begins publication of Anfangsgründe der Deich und Wasser-Baukunst ("Principles of Dike and Aquatic Engineering") advocating scientific recording of tides.<br><br>Mathematics<br> Joshua Kirby publishes the pamph""",
"""Elmhurst Hospital seeks new dogs for therapy program<br><br>Elmhurst Hospital is seeking more dogs for its animal-assisted therapy program. Teams of dogs and handlers have made nearly 11,000 patient visits since the program began in 2012.<br><br>ELMHURST – Elmhurst Hospital is seeking more dogs for its animal-assisted therapy program.<br><br>To be considered, dogs must meet the following requirements: sit, stay and more on command, walk loosely on a leash without pulling, get along well with other dogs, perform required commands without treats, like people, not be overly vocal, be at least 1 year old""",
"""List of United States university campuses by undergraduate enrollment<br><br>This list of largest United States universities by undergraduate enrollment includes only individual four-year campuses, not four-year universities.  Universities can have multiple campuses with a single administration.<br><br>What this list includes:<br>A single Individual campus with a single physical location of a four-year public university within the United States<br>Enrollment is the sum of the headcount of undergraduate students<br>Enrollment is counted by the 21st-day headcount, as provided to the United States Department of Education under the Common Data Set program.<br>Campuses that have small secondary physical"""
]

In [51]:
s_list = [s.replace("<br>", "\n") for s in s_list]

In [52]:
hook_pt = 'blocks.3.hook_attn_out'
ae = suite.module_dict[f'attn_3'].ae

In [53]:
_, cache = model.run_with_cache(s_list, return_type="loss", names_filter=[hook_pt])
x = cache[hook_pt]
f = ae.encode(x)
y = ae.decode(f)

In [54]:
(y-x).pow(2).sum() / (x - x.mean([0, 1])).pow(2).sum()

tensor(0.1188, device='cuda:0')

In [55]:
a = f[0, :, 0]
str_toks = model.to_tokens(s)[0, :]
for act, tok in zip(a, str_toks):
    x = "   <------------------" if act>0 else ""
    print(f"{act} {model.tokenizer.decode(tok)} {x}")


0.0 <|endoftext|> 
0.09912377595901489 The    <------------------
0.23582345247268677  educational    <------------------
0.13851162791252136  aim    <------------------
0.43569016456604004  of    <------------------
0.728547990322113  the    <------------------
0.3769438862800598  proposed    <------------------
0.2572154402732849  K    <------------------
0.0 01 
0.0  proposal 
0.0  is 
0.28884175419807434  to    <------------------
0.7897344827651978  allow    <------------------
0.700764536857605  the    <------------------
0.0  applicant 
0.41991740465164185  to    <------------------
0.5291426181793213  train    <------------------
0.4033319056034088  in    <------------------
0.32827842235565186  developmental    <------------------
0.0  affective 
0.0  neuro 
0.31451416015625 science    <------------------
0.8079363107681274  and    <------------------
0.5762215852737427  pediatric    <------------------
0.4472712278366089  bipolar    <------------------
0.4401834309101105  dis

In [147]:
(a!=0).sum()

tensor(0, device='cuda:0')